# DA LMP Forecast Model Investigation

Investigation of the PJM Western Hub Day-Ahead LMP forecast model.  
Model: LightGBM quantile regression with asinh variance-stabilizing transformation.  
Target: `lmp_total_target` (asinh-transformed DA LMP for Western Hub).

**Sections:**
1. Setup & Data Loading
2. Feature Analysis
3. Model Training & Forecast
4. Error Analysis
5. Visualizations

---
## Section 1: Setup & Data Loading

In [ ]:
import os
import sys
import warnings

# Ensure working directory is da-model/ so that `src.*` imports resolve
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, os.getcwd())

# Initialize logging and env vars (same as forecast.py __main__)
import src.settings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta

from src.pjm_da_forecast import configs
from src.pjm_da_forecast.features.builder import build_features
from src.pjm_da_forecast.features.preprocessing import asinh_transform, asinh_inverse
from src.pjm_da_forecast.models.lightgbm_quantile import LightGBMQuantile
from src.pjm_da_forecast.evaluation.metrics import evaluate_forecast
from src.pjm_da_forecast.pipelines.forecast import run as run_forecast, NON_FEATURE_COLS

# Plot style
sns.set_theme(style="darkgrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 100,
})

print("Setup complete.")

### Build the feature matrix

Pull all data from the database and construct the full feature matrix using `build_features()`.  
Mode `full_feature` uses data from 2020+ and includes DA load features.

In [ ]:
df = build_features(mode="full_feature")
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Total rows: {len(df):,}")

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
print(f"Feature count: {len(feature_cols)}")

# Missing values summary
missing = df[feature_cols].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0].sort_values("missing_pct", ascending=False)

if len(missing_report) > 0:
    print(f"\nFeatures with missing values: {len(missing_report)}")
    display(missing_report.head(20))
else:
    print("\nNo missing values in any feature column.")

---
## Section 2: Feature Analysis

### Feature columns and types

List all features grouped by category (LMP lags, gas, load, calendar).

In [ ]:
# Categorize features by prefix
categories = {
    "LMP lag (DA)": [c for c in feature_cols if c.startswith("da_lmp_lag")],
    "LMP rolling/shape": [c for c in feature_cols if c.startswith("lmp_") and not c.startswith("lmp_total")],
    "RT LMP": [c for c in feature_cols if c.startswith("rt_lmp")],
    "DART spread": [c for c in feature_cols if c.startswith("dart_")],
    "Gas": [c for c in feature_cols if c.startswith("gas_") or c == "implied_heat_rate"],
    "DA load": [c for c in feature_cols if c.startswith("da_load")],
    "RT load": [c for c in feature_cols if c.startswith("rt_load")],
    "Load forecast error": [c for c in feature_cols if "load_forecast" in c],
    "Calendar/temporal": [c for c in feature_cols if any(c.startswith(p) for p in
        ["hour_", "dow_", "month_", "annual_", "weekly_", "is_", "summer_"])],
}

# Catch uncategorized
categorized = set()
for cols in categories.values():
    categorized.update(cols)
categories["Other"] = [c for c in feature_cols if c not in categorized]

for cat, cols in categories.items():
    if cols:
        print(f"\n{cat} ({len(cols)} features):")
        # Show first 10, then ellipsis if more
        display_cols = cols[:10]
        for c in display_cols:
            dtype = df[c].dtype
            print(f"  {c:<50s} {str(dtype):<10s}")
        if len(cols) > 10:
            print(f"  ... and {len(cols) - 10} more")

print(f"\nTotal features: {len(feature_cols)}")

### Correlation with target

Top features correlated with `lmp_total_target` (asinh-transformed DA LMP).

In [ ]:
# Correlation of each feature with the target
corr_with_target = df[feature_cols + ["lmp_total_target"]].corr()["lmp_total_target"].drop("lmp_total_target")
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

# Show top 30
top_n = 30
top_corr = corr_with_target.loc[corr_sorted.head(top_n).index]

fig, ax = plt.subplots(figsize=(10, 8))
top_corr.sort_values().plot.barh(ax=ax, color=[
    "#2ecc71" if v > 0 else "#e74c3c" for v in top_corr.sort_values()
])
ax.set_xlabel("Pearson correlation with lmp_total_target")
ax.set_title(f"Top {top_n} Features by Correlation with Target")
ax.axvline(x=0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"\nTop 10 positively correlated features:")
display(corr_with_target.sort_values(ascending=False).head(10).round(4).to_frame("correlation"))
print(f"\nTop 10 negatively correlated features:")
display(corr_with_target.sort_values(ascending=True).head(10).round(4).to_frame("correlation"))

### Feature importance from a trained model

Train a LightGBM model on the full dataset and extract built-in feature importance (split-based).  
Also attempt SHAP values if `shap` is available.

In [ ]:
# Train a model on full data for importance analysis
X_all = df[feature_cols].astype(float)
y_all = df["lmp_total_target"].astype(float)

model_for_importance = LightGBMQuantile()
model_for_importance.fit(X_all, y_all)

# LightGBM built-in importance (from median quantile model)
importance_df = model_for_importance.get_feature_importance()

fig, ax = plt.subplots(figsize=(10, 8))
top_imp = importance_df.head(30)
ax.barh(top_imp["feature"][::-1], top_imp["importance"][::-1], color="#3498db")
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("Top 30 Features — LightGBM Built-in Importance (Median Model)")
plt.tight_layout()
plt.show()

display(importance_df.head(20))

In [ ]:
# SHAP analysis (if available)
try:
    import shap

    # Use a sample for SHAP (full dataset can be slow)
    sample_size = min(2000, len(X_all))
    X_sample = X_all.sample(n=sample_size, random_state=42)

    # Use the point forecast model (MSE objective) for SHAP
    explainer = shap.TreeExplainer(model_for_importance.point_model)
    shap_values = explainer.shap_values(X_sample)

    fig, ax = plt.subplots(figsize=(10, 10))
    shap.summary_plot(shap_values, X_sample, max_display=25, show=False)
    plt.title("SHAP Summary Plot (Top 25 Features)")
    plt.tight_layout()
    plt.show()

    # Mean absolute SHAP values
    mean_shap = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)

    print("Top 15 features by mean |SHAP|:")
    display(mean_shap.head(15))

except ImportError:
    print("shap not installed — skipping SHAP analysis.")
    print("Install with: pip install shap")
except Exception as e:
    print(f"SHAP analysis failed: {e}")

### Target variable distribution

Distribution of the target `lmp_total_target` in both raw ($/MWh) and asinh-transformed space.

In [ ]:
target_asinh = df["lmp_total_target"]
target_raw = asinh_inverse(target_asinh)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(target_raw, bins=100, color="#e74c3c", alpha=0.7, edgecolor="black", linewidth=0.3)
axes[0].set_xlabel("DA LMP ($/MWh)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Distribution — Raw ($/MWh)")
axes[0].axvline(target_raw.median(), color="black", linestyle="--", label=f"Median: ${target_raw.median():.1f}")
axes[0].axvline(target_raw.mean(), color="blue", linestyle="--", label=f"Mean: ${target_raw.mean():.1f}")
axes[0].legend()

# Asinh-transformed distribution
axes[1].hist(target_asinh, bins=100, color="#3498db", alpha=0.7, edgecolor="black", linewidth=0.3)
axes[1].set_xlabel("asinh(DA LMP)")
axes[1].set_ylabel("Count")
axes[1].set_title("Target Distribution — asinh Transformed")
axes[1].axvline(target_asinh.median(), color="black", linestyle="--", label=f"Median: {target_asinh.median():.2f}")
axes[1].axvline(target_asinh.mean(), color="blue", linestyle="--", label=f"Mean: {target_asinh.mean():.2f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Raw target stats:")
display(target_raw.describe().round(2).to_frame("$/MWh"))
print(f"\nasinh target stats:")
display(target_asinh.describe().round(4).to_frame("asinh($/MWh)"))

---
## Section 3: Model Training & Forecast

Run the full forecast pipeline for a target date and examine outputs.

In [ ]:
# Determine the forecast date: use 2026-02-25 if available, else latest date in data
available_dates = sorted(df["date"].unique())
target_date = date(2026, 2, 25)
if target_date not in available_dates:
    target_date = available_dates[-1]
    print(f"2026-02-25 not in data. Using latest available date: {target_date}")
else:
    print(f"Using target date: {target_date}")

FORECAST_DATE = str(target_date)

In [ ]:
# Run the forecast pipeline
result = run_forecast(forecast_date=FORECAST_DATE, mode="full_feature")

### Output table: Actual vs Forecast

In [ ]:
output_table = result["output_table"]
display(output_table.style.format({
    col: "${:.2f}" for col in output_table.columns
    if col not in ["Date", "Type"]
}).set_caption("DA LMP Forecast — Western Hub ($/MWh)"))

### Quantile bands

In [ ]:
quantiles_table = result["quantiles_table"]
display(quantiles_table.style.format({
    col: "${:.2f}" for col in quantiles_table.columns
    if col not in ["Date", "Type"]
}).set_caption("Quantile Bands ($/MWh)"))

### Evaluation metrics

In [ ]:
metrics = result.get("metrics")
if metrics:
    # Display key metrics
    key_metrics = {
        "MAE ($/MWh)": f"${metrics.get('mae', 0):.2f}",
        "RMSE ($/MWh)": f"${metrics.get('rmse', 0):.2f}",
        "MAPE (%)": f"{metrics.get('mape', 0):.1f}%",
        "rMAE (vs naive-7d)": f"{metrics.get('rmae', 'N/A')}",
        "Mean Pinball Loss": f"{metrics.get('mean_pinball', 0):.4f}",
        "CRPS": f"{metrics.get('crps', 0):.4f}",
        "Coverage 80% PI": f"{metrics.get('coverage_80pct', 0):.0%}",
        "Coverage 90% PI": f"{metrics.get('coverage_90pct', 0):.0%}",
        "Coverage 98% PI": f"{metrics.get('coverage_98pct', 0):.0%}",
        "Sharpness 80% PI ($/MWh)": f"${metrics.get('sharpness_80pct', 0):.2f}",
        "Sharpness 90% PI ($/MWh)": f"${metrics.get('sharpness_90pct', 0):.2f}",
    }
    metrics_df = pd.DataFrame(list(key_metrics.items()), columns=["Metric", "Value"])
    display(metrics_df.style.hide(axis="index").set_caption(f"Forecast Metrics for {FORECAST_DATE}"))
else:
    print("No actuals available for this date — metrics cannot be computed.")

---
## Section 4: Error Analysis

Analyze forecast errors across multiple dimensions: hourly profile, day-of-week, distribution, and time series.

We run a backtest over the most recent 30 days (or as many as available) to get a meaningful error sample.

In [ ]:
# Backtest: run forecast for the most recent N dates
# We reuse the already-loaded feature matrix to avoid repeated DB calls.
# Manually replicate the forecast loop from forecast.py.

BACKTEST_DAYS = 30

# Pick the last BACKTEST_DAYS dates that have 24 hours of data
date_counts = df.groupby("date").size()
full_dates = sorted(date_counts[date_counts == 24].index)

# Need at least some training data before the first backtest date
backtest_dates = full_dates[-BACKTEST_DAYS:]
print(f"Backtest period: {backtest_dates[0]} to {backtest_dates[-1]} ({len(backtest_dates)} days)")

backtest_results = []

for bd in backtest_dates:
    df_train = df[df["date"] < bd].copy()
    df_test = df[df["date"] == bd].copy()

    if len(df_train) < 100 or len(df_test) != 24:
        continue

    fc = [c for c in df_train.columns if c not in NON_FEATURE_COLS]
    X_tr = df_train[fc].astype(float)
    y_tr = df_train["lmp_total_target"].astype(float)
    X_te = df_test[fc].astype(float)

    model = LightGBMQuantile()
    model.fit(X_tr, y_tr)
    preds_asinh = model.predict(X_te)

    # Inverse transform
    preds = preds_asinh.copy()
    q_cols = [c for c in preds.columns if c.startswith("q_")]
    for col in q_cols:
        preds[col] = asinh_inverse(preds_asinh[col])
    if "point_forecast" in preds.columns:
        preds["point_forecast"] = asinh_inverse(preds_asinh["point_forecast"])

    actuals_raw = asinh_inverse(df_test["lmp_total_target"].values)
    hours = df_test["hour_ending"].astype(int).values

    for i, h in enumerate(hours):
        backtest_results.append({
            "date": bd,
            "hour_ending": h,
            "actual": actuals_raw[i],
            "forecast": preds["point_forecast"].values[i],
            "q_10": preds["q_0.10"].values[i],
            "q_90": preds["q_0.90"].values[i],
            "q_05": preds["q_0.05"].values[i],
            "q_95": preds["q_0.95"].values[i],
        })

bt = pd.DataFrame(backtest_results)
bt["error"] = bt["forecast"] - bt["actual"]
bt["abs_error"] = bt["error"].abs()
bt["pct_error"] = (bt["error"] / bt["actual"].replace(0, np.nan)) * 100
bt["date_dt"] = pd.to_datetime(bt["date"])
bt["dow"] = bt["date_dt"].dt.dayofweek
bt["dow_name"] = bt["date_dt"].dt.day_name()

print(f"Backtest complete: {len(bt)} hourly observations across {bt['date'].nunique()} days")
print(f"Overall MAE: ${bt['abs_error'].mean():.2f}/MWh")
print(f"Overall MAPE: {bt['pct_error'].abs().mean():.1f}%")
print(f"Mean Error (bias): ${bt['error'].mean():.2f}/MWh")

### Hourly error profile

Which hour-ending slots have the largest forecast errors?

In [ ]:
hourly_stats = bt.groupby("hour_ending").agg(
    mean_error=("error", "mean"),
    mae=("abs_error", "mean"),
    std_error=("error", "std"),
    median_error=("error", "median"),
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE by hour
axes[0].bar(hourly_stats.index, hourly_stats["mae"], color="#3498db", alpha=0.8)
axes[0].set_xlabel("Hour Ending")
axes[0].set_ylabel("MAE ($/MWh)")
axes[0].set_title("Mean Absolute Error by Hour")
axes[0].set_xticks(range(1, 25))

# Bias (mean error) by hour
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in hourly_stats["mean_error"]]
axes[1].bar(hourly_stats.index, hourly_stats["mean_error"], color=colors, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Hour Ending")
axes[1].set_ylabel("Mean Error ($/MWh)")
axes[1].set_title("Forecast Bias by Hour (positive = over-prediction)")
axes[1].set_xticks(range(1, 25))

plt.tight_layout()
plt.show()

display(hourly_stats)

### Error by day of week

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_stats = bt.groupby("dow_name").agg(
    mean_error=("error", "mean"),
    mae=("abs_error", "mean"),
    mape=("pct_error", lambda x: x.abs().mean()),
    count=("error", "count"),
).round(2)
dow_stats = dow_stats.reindex(dow_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(dow_stats.index, dow_stats["mae"], color="#9b59b6", alpha=0.8)
axes[0].set_ylabel("MAE ($/MWh)")
axes[0].set_title("MAE by Day of Week")
axes[0].tick_params(axis="x", rotation=45)

colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in dow_stats["mean_error"]]
axes[1].bar(dow_stats.index, dow_stats["mean_error"], color=colors, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Mean Error ($/MWh)")
axes[1].set_title("Forecast Bias by Day of Week")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

display(dow_stats)

### Error distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error histogram
axes[0].hist(bt["error"], bins=50, color="#3498db", alpha=0.7, edgecolor="black", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1)
axes[0].axvline(bt["error"].mean(), color="orange", linestyle="--",
                label=f"Mean: ${bt['error'].mean():.2f}")
axes[0].axvline(bt["error"].median(), color="green", linestyle="--",
                label=f"Median: ${bt['error'].median():.2f}")
axes[0].set_xlabel("Forecast Error ($/MWh)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Forecast Errors (Forecast - Actual)")
axes[0].legend()

# Absolute error histogram
axes[1].hist(bt["abs_error"], bins=50, color="#e67e22", alpha=0.7, edgecolor="black", linewidth=0.3)
axes[1].axvline(bt["abs_error"].mean(), color="red", linestyle="--",
                label=f"MAE: ${bt['abs_error'].mean():.2f}")
axes[1].axvline(bt["abs_error"].median(), color="green", linestyle="--",
                label=f"Median: ${bt['abs_error'].median():.2f}")
axes[1].set_xlabel("Absolute Error ($/MWh)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Absolute Errors")
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print("Error distribution summary:")
display(bt["error"].describe().round(2).to_frame("error ($/MWh)"))

# Bias direction
n_over = (bt["error"] > 0).sum()
n_under = (bt["error"] < 0).sum()
print(f"\nBias direction: Over-predicted {n_over} hours ({n_over/len(bt)*100:.1f}%), "
      f"Under-predicted {n_under} hours ({n_under/len(bt)*100:.1f}%)")
bias = "over-prediction" if bt["error"].mean() > 0 else "under-prediction"
print(f"Systematic bias: {bias} (mean error = ${bt['error'].mean():.2f}/MWh)")

### Time series of daily errors over the backtest period

In [ ]:
daily_bt = bt.groupby("date").agg(
    mae=("abs_error", "mean"),
    mean_error=("error", "mean"),
    mape=("pct_error", lambda x: x.abs().mean()),
).reset_index()
daily_bt["date_dt"] = pd.to_datetime(daily_bt["date"])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Daily MAE
axes[0].bar(daily_bt["date_dt"], daily_bt["mae"], color="#3498db", alpha=0.8, width=0.8)
axes[0].axhline(daily_bt["mae"].mean(), color="red", linestyle="--",
                label=f"Avg MAE: ${daily_bt['mae'].mean():.2f}")
axes[0].set_ylabel("Daily MAE ($/MWh)")
axes[0].set_title("Daily MAE Over Backtest Period")
axes[0].legend()

# Daily bias (mean error)
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in daily_bt["mean_error"]]
axes[1].bar(daily_bt["date_dt"], daily_bt["mean_error"], color=colors, alpha=0.8, width=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Mean Error ($/MWh)")
axes[1].set_title("Daily Forecast Bias (positive = over-prediction)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Table of worst days
worst_days = daily_bt.nlargest(5, "mae")[["date", "mae", "mean_error", "mape"]]
worst_days.columns = ["Date", "MAE ($/MWh)", "Mean Error ($/MWh)", "MAPE (%)"]
print("Top 5 worst forecast days:")
display(worst_days.round(2))

---
## Section 5: Visualizations

### Actual vs Forecast line chart (target date)

In [ ]:
# Extract hourly values from the output table for the target date
ot = result["output_table"]
hours = list(range(1, 25))

actual_row = ot[ot["Type"] == "Actual"]
forecast_row = ot[ot["Type"] == "Forecast"]

fig, ax = plt.subplots(figsize=(14, 6))

if len(actual_row) > 0:
    actuals = [actual_row.iloc[0][f"HE{h}"] for h in hours]
    ax.plot(hours, actuals, "o-", color="#2ecc71", linewidth=2, markersize=5, label="Actual")

if len(forecast_row) > 0:
    forecasts = [forecast_row.iloc[0][f"HE{h}"] for h in hours]
    ax.plot(hours, forecasts, "s--", color="#e74c3c", linewidth=2, markersize=5, label="Forecast")

ax.set_xlabel("Hour Ending")
ax.set_ylabel("DA LMP ($/MWh)")
ax.set_title(f"Actual vs Forecast — Western Hub — {FORECAST_DATE}")
ax.set_xticks(hours)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Prediction interval fan chart (quantile bands)

Shows the forecast point estimate with 80%, 90%, and 98% prediction intervals as shaded bands.

In [ ]:
qt = result["quantiles_table"]

# Extract quantile bands by hour
def get_quantile_hourly(qt_df, ptype):
    row = qt_df[qt_df["Type"] == ptype]
    if len(row) == 0:
        return [np.nan] * 24
    return [row.iloc[0][f"HE{h}"] for h in range(1, 25)]

p01 = get_quantile_hourly(qt, "P01")
p05 = get_quantile_hourly(qt, "P05")
p10 = get_quantile_hourly(qt, "P10")
p25 = get_quantile_hourly(qt, "P25")
p50 = get_quantile_hourly(qt, "P50")
p75 = get_quantile_hourly(qt, "P75")
p90 = get_quantile_hourly(qt, "P90")
p95 = get_quantile_hourly(qt, "P95")
p99 = get_quantile_hourly(qt, "P99")

fig, ax = plt.subplots(figsize=(14, 7))

# Fan chart: wider bands are more transparent
ax.fill_between(hours, p01, p99, alpha=0.10, color="#3498db", label="98% PI (P01-P99)")
ax.fill_between(hours, p05, p95, alpha=0.15, color="#3498db", label="90% PI (P05-P95)")
ax.fill_between(hours, p10, p90, alpha=0.20, color="#3498db", label="80% PI (P10-P90)")
ax.fill_between(hours, p25, p75, alpha=0.30, color="#3498db", label="50% PI (P25-P75)")

# Forecast (point)
if len(forecast_row) > 0:
    ax.plot(hours, forecasts, "s-", color="#e74c3c", linewidth=2, markersize=4, label="Point Forecast", zorder=5)

# Actuals
if len(actual_row) > 0:
    ax.plot(hours, actuals, "o-", color="#2ecc71", linewidth=2.5, markersize=6, label="Actual", zorder=6)

ax.set_xlabel("Hour Ending")
ax.set_ylabel("DA LMP ($/MWh)")
ax.set_title(f"Prediction Interval Fan Chart — Western Hub — {FORECAST_DATE}")
ax.set_xticks(hours)
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Feature importance bar chart (from Section 2 model)

In [ ]:
# Normalized feature importance (top 20)
top20 = importance_df.head(20).copy()
top20["importance_pct"] = (top20["importance"] / top20["importance"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20["feature"][::-1], top20["importance_pct"][::-1], color="#2980b9", alpha=0.85)

# Add value labels
for bar, val in zip(bars, top20["importance_pct"][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)

ax.set_xlabel("Importance (% of top 20 total splits)")
ax.set_title("Top 20 Features — LightGBM Split-Based Importance")
plt.tight_layout()
plt.show()

### Scatter plot: Actual vs Predicted (backtest period)

Each point is one hourly observation from the backtest. Perfect forecast lies on the 45-degree line.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(bt["actual"], bt["forecast"], alpha=0.4, s=15, color="#3498db", edgecolors="none")

# 45-degree line
lims = [
    min(bt["actual"].min(), bt["forecast"].min()) - 5,
    max(bt["actual"].max(), bt["forecast"].max()) + 5,
]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect forecast (y=x)")

# Best fit line
from numpy.polynomial.polynomial import polyfit
b, m = polyfit(bt["actual"], bt["forecast"], 1)
x_fit = np.linspace(lims[0], lims[1], 100)
ax.plot(x_fit, b + m * x_fit, "--", color="orange", linewidth=1,
        label=f"Best fit: y = {m:.2f}x + {b:.1f}")

ax.set_xlabel("Actual DA LMP ($/MWh)")
ax.set_ylabel("Forecast DA LMP ($/MWh)")
ax.set_title(f"Actual vs Predicted — Backtest ({bt['date'].nunique()} days)")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# R-squared
ss_res = ((bt["actual"] - bt["forecast"]) ** 2).sum()
ss_tot = ((bt["actual"] - bt["actual"].mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print(f"R-squared: {r2:.4f}")